In [ ]:
#pip install scikit-learn xgboost lightgbm joblib pandas numpy

In [1]:
import re
import os
import json
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier


In [2]:
def  process_data(df):
    df = convert_to_datetime(df)
    df, html_df = handle_missing_data(df)
    return df

def is_integer_like(series):
    return pd.api.types.is_numeric_dtype(series) and \
        series.dropna().apply(lambda x: float(x).is_integer()).all()

def handle_missing_data(df):
    try:
        df = df.copy()

        ignore_types = ['object', 'string', 'timedelta', 'complex']
        ignored_columns_info = {}

        # Drop columns entirely missing
        df = df.dropna(axis=1, how='all')

        # Identify column types ONCE
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
        bool_cols = df.select_dtypes(include=['bool']).columns
        category_cols = df.select_dtypes(include=['category']).columns
        ignored_cols = df.select_dtypes(include=ignore_types).columns

        int_like_cols = [col for col in numeric_cols if is_integer_like(df[col])]

        for col in ignored_cols:
            ignored_columns_info[col] = "Ignored because of optional data"

        # Initialize imputed flags
        imputed_flags = pd.DataFrame(False, index=df.index, columns=df.columns)

        # Numeric columns
        if len(numeric_cols) > 0:
            imputed_flags[numeric_cols] = df[numeric_cols].isna()

            imputer = KNNImputer(n_neighbors=5)
            imputed_numeric = imputer.fit_transform(df[numeric_cols])

            imputed_numeric_df = pd.DataFrame(
                imputed_numeric,
                columns=numeric_cols,
                index=df.index
            ).round(2)

            for col in int_like_cols:
                imputed_numeric_df[col] = imputed_numeric_df[col].round().astype("Int64")

            df[numeric_cols] = imputed_numeric_df

        # Category columns
        for col in category_cols:
            mask = df[col].isna()
            if mask.any():
                imputed_flags.loc[mask, col] = True
                mode_val = df[col].mode(dropna=True)
                fill_val = mode_val.iloc[0] if not mode_val.empty else "Unknown"
                df[col] = df[col].fillna(fill_val)

        # Boolean columns
        for col in bool_cols:
            mask = df[col].isna()
            if mask.any():
                imputed_flags.loc[mask, col] = True
                df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])

        # Datetime columns
        for col in datetime_cols:
            df[col] = pd.to_datetime(df[col], errors='coerce')

            mask = df[col].isna()
            if not mask.any():
                continue

            diffs = df[col].diff().dropna()
            if diffs.empty:
                continue

            avg_diff_sec = diffs.mean().total_seconds()

            if avg_diff_sec < 3600:
                offset = pd.to_timedelta(avg_diff_sec, unit='s')
            elif avg_diff_sec < 86400:
                offset = pd.to_timedelta(avg_diff_sec, unit='s')
            elif avg_diff_sec < 2629746:
                offset = pd.DateOffset(days=round(avg_diff_sec / 86400))
            elif avg_diff_sec < 31557600:
                offset = pd.DateOffset(months=round(avg_diff_sec / 2629746))
            else:
                offset = pd.DateOffset(years=round(avg_diff_sec / 31557600))

            for idx in df[mask].index:
                prev_idx = df.index.get_loc(idx) - 1
                if prev_idx >= 0:
                    df.at[idx, col] = df.iloc[prev_idx][col] + offset
                    imputed_flags.at[idx, col] = True

        # Prepare JSON output
        data = []
        for idx, row in df.iterrows():
            row_data = {}
            for col in df.columns:
                val = row[col]
                row_data[col] = {
                    "value": val.strftime('%Y-%m-%d %H:%M:%S') if isinstance(val, pd.Timestamp) else val,
                    "is_imputed": bool(imputed_flags.at[idx, col])
                }
            data.append(row_data)
        return df, data

    except Exception as e:
        print("Error:", e)
        return None, None

def detect_and_parse_date(value):
    """
    Detects and converts dates in multiple formats, including:
    - MM-DD-YYYY
    - DD-MM-YYYY
    - MM/DD/YYYY
    - DD/MM/YYYY
    - YYYY-MM-DD
    - YYYY/MM/DD
    Supports optional time: HH:MM or HH:MM:SS.
    """
    if pd.isna(value) or not isinstance(value, str) or value.strip() == "":
        return pd.NaT

    value = value.strip()
    # YYYY-MM-DD or YYYY/MM/DD (ISO-like)
    if re.match(r"^\d{4}[-/]\d{2}[-/]\d{2}([ T]\d{2}:\d{2}(:\d{2})?)?$", value):
        try:
            return dateutil.parser.parse(value, yearfirst=True)
        except:
            pass

    # DD-MM-YYYY or DD/MM/YYYY (Europe/India)
    if re.match(r"^\d{2}[-/]\d{2}[-/]\d{4}([ T]\d{2}:\d{2}(:\d{2})?)?$", value):
        try:
            return dateutil.parser.parse(value, dayfirst=True)
        except:
            pass

    # MM-DD-YYYY or MM/DD/YYYY (US style)
    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})([ T]\d{2}:\d{2}(:\d{2})?)?$", value)
    if m:
        month, day = int(m.group(1)), int(m.group(2))
        if month <= 12 and day <= 31:
            try:
                return dateutil.parser.parse(value, dayfirst=False)
            except:
                pass
    try:
        return dateutil.parser.parse(value)
    except:
        return pd.NaT



def convert_to_datetime(df):
    """
    Converts object (string) columns containing dates to datetime format.
    """
    for col in df.columns:
        if df[col].dtype == "object":  # Process only string columns
            if df[col].str.contains(r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}", na=False).any():
                df[col] = df[col].apply(detect_and_parse_date)
    return df

def detect_task_type(y: pd.Series, numeric_threshold=0.75):
    """
    Robust regression vs classification detection.
    Designed to NEVER misclassify true regression as classification.
    """

    try:
        if y is None or y.empty:
            return "classification"

        y_non_null = y.dropna()
        if y_non_null.empty:
            return "classification"

        # Try numeric conversion (CSV-safe)
        y_numeric = pd.to_numeric(y_non_null, errors="coerce")
        numeric_ratio = y_numeric.notna().mean()

        # If NOT mostly numeric → classification
        if numeric_ratio < numeric_threshold:
            return "classification"

        # From here: target is numeric
        y_clean = y_numeric.dropna()

        unique_vals = y_clean.unique()
        unique_count = len(unique_vals)

        # STRICT classification checks ONLY
        # Binary labels
        if unique_count == 2:
            return "classification"

        # Small integer labels (0/1/2/3 etc.)
        if unique_count <= 10 and y_clean.min() >= 0 and y_clean.max() <= unique_count:
            return "classification"

        # Everything else → regression
        return "regression"

    except Exception as e:
        print("Task detection fallback:", e)
        return "regression"




def build_preprocessor(X):
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

def safe_json_dump(obj, file_path):
    import json
    import numpy as np

    def replace_nan(o):
        if isinstance(o, dict):
            return {k: replace_nan(v) for k, v in o.items()}
        if isinstance(o, list):
            return [replace_nan(v) for v in o]
        if isinstance(o, float) and np.isnan(o):
            return None
        return o

    with open(file_path, "w") as f:
        json.dump(replace_nan(obj), f, indent=4)


def train_ml_models(
    df: pd.DataFrame,
    target_col: str,
    task_type: str,
    model_dir: str
):
    try:
        os.makedirs(model_dir, exist_ok=True)

        df = df[df[target_col].notna()].copy()
        if df.empty:
            return {"status": False, "message": "Target column empty after NaN removal"}

        X = df.drop(columns=[target_col])
        y = df[target_col]

        preprocessor = build_preprocessor(X)

        label_encoder = None

        if task_type == "classification":
            label_encoder = LabelEncoder()
            y = label_encoder.fit_transform(y)

        X_train, X_val, y_train, y_val = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            stratify=y if task_type == "classification" else None
        )

        if task_type == "regression":
            models = {
                "LinearRegression": LinearRegression(),
                "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
                "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
                "LightGBM": LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
            }
            metric_fn = lambda y_true, y_pred: mean_squared_error(y_true, y_pred, squared=False)
            metric_name = "RMSE"
            best_score = float("inf")
            is_better = lambda s, b: s < b
        else:
            models = {
                "LogisticRegression": LogisticRegression(max_iter=1000),
                "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
                "LightGBM": LGBMClassifier()
            }
            metric_fn = lambda y_true, y_pred: f1_score(y_true, y_pred, average="weighted")
            metric_name = "F1"
            best_score = -1
            is_better = lambda s, b: s > b

        registry = {
            "task_type": task_type,
            "metric_type": metric_name,
            "best_model": None,
            "models": {}
        }

        for name, model in models.items():
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", model)
            ])

            pipeline.fit(X_train, y_train)
            preds = pipeline.predict(X_val)
            score = round(metric_fn(y_val, preds), 4)

            # Save model
            model_bundle = {
                "pipeline": pipeline,
                "label_encoder": label_encoder
            }
            
            joblib.dump(model_bundle, os.path.join(model_dir, f"{name}.pkl"))


            registry["models"][name] = {
                "input": {
                    "hyperparameters": model.get_params(),
                    "train_size": len(X_train),
                    "validation_size": len(X_val)
                },
                "output": {
                    metric_name: score
                }
            }

            if is_better(score, best_score):
                best_score = score
                registry["best_model"] = name

        # Save single registry file
        safe_json_dump(registry, os.path.join(model_dir, "metrics.json"))



        return {
            "status": True,
            "best_model": registry["best_model"],
            "metric": registry["models"][registry["best_model"]]["output"][metric_name],
            "all_models": registry["models"]
        }

    except Exception as e:
        return {"status": False, "message": str(e)}

def load_model(model_dir: str):
    """
    Loads the selected model based on metrics.json
    """
    metrics_path = os.path.join(model_dir, "metrics.json")
    if not os.path.exists(metrics_path):
        raise FileNotFoundError("metrics.json not found")

    with open(metrics_path, "r") as f:
        registry = json.load(f)

    model_name = registry.get("best_model")
    if not model_name:
        raise ValueError("No model selected in metrics.json")

    model_path = os.path.join(model_dir, f"{model_name}.pkl")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"{model_name}.pkl not found")

    model_bundle = joblib.load(model_path)

    pipeline = model_bundle["pipeline"]
    label_encoder = model_bundle.get("label_encoder")

    return pipeline, label_encoder, registry



def predict(model_dir: str, input_df: pd.DataFrame):
    """
    Loads model and returns predictions
    """
    model, label_encoder, registry = load_model(model_dir)

    predictions = model.predict(input_df)

    if label_encoder:
        preds = label_encoder.inverse_transform(predictions)
    else:
        preds = predictions
    
    response = {
        "predictions": preds.tolist()
    }

    # Add probabilities only if classification
    if registry["task_type"] == "classification":
        if hasattr(model.named_steps["model"], "predict_proba"):
            probs = model.predict_proba(input_df)

            # Optional: map probabilities to class names
            if label_encoder is not None:
                class_labels = label_encoder.inverse_transform(
                    range(len(label_encoder.classes_))
                )
                response["probabilities"] = [
                    dict(zip(class_labels, p)) for p in probs
                ]
            else:
                response["probabilities"] = probs.tolist()


    return response



In [3]:
target_col = "Pressure"
file_path=r"G:\F\DIGIOTAI\data\final_truck_sensor_data_updatedmod.csv"
model_dir="models/truck"
data=pd.read_csv(file_path)
data.isnull().sum()

Timestamp      0
DeviceID       0
SensorID       0
Speed          0
Pressure       0
Temperature    0
Wear           0
Status         0
Obs_Obj        0
Collision      0
Type           0
dtype: int64

In [ ]:
#data=process_data(data)
#data.isnull().sum()

In [ ]:
model_type = detect_task_type(data[target_col])

print("Detected task type:", model_type)

res = train_ml_models(
    df=data,
    target_col=target_col,
    task_type=model_type,
    model_dir=model_dir
)

print(res)


In [5]:
predict(model_dir, data[:5])

{'predictions': [1500.6978131150672,
  1394.0000000000607,
  1150.0000000000587,
  1503.7693584488875,
  1271.9999999999375]}